## 1. Import libraries

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster

## 2. Load cleaned dataset

In [ ]:
df = pd.read_csv(
    '../data/cleaned/dataset_maderas_final.csv',
    sep=';',
    encoding='utf-8-sig'
)

In [ ]:
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['tree_quantity'] = pd.to_numeric(df['tree_quantity'], errors='coerce')

## 3. Create output folder for maps

In [ ]:
os.makedirs('../img/maps', exist_ok=True)

## 4. Extract provinces and municipalities from dataset

In [ ]:
dataset_provinces = sorted(df['province'].dropna().unique().tolist())
dataset_municipalities = sorted(df['municipality'].dropna().unique().tolist())

print("Provinces in dataset:")
print(dataset_provinces)

print("\nNumber of municipalities in dataset:")
print(len(dataset_municipalities))

## 5. Load Andalusia autonomous boundary

In [ ]:
andalusia = gpd.read_file(
    "../data/boundaries/lineas_limite/SHP_ETRS89/recintos_autonomicas_inspire_peninbal_etrs89/recintos_autonomicas_inspire_peninbal_etrs89.shp"
)

In [ ]:
andalusia = andalusia.to_crs(epsg=4326)
andalusia.columns.tolist()

In [ ]:
andalusia[['NAMEUNIT']].head(20)

In [ ]:
andalusia = andalusia[
    andalusia['NAMEUNIT'].str.upper().str.contains('ANDALUC')
].copy()

andalusia

## 6. Load and filter provincial boundaries

In [ ]:
provinces = gpd.read_file(
    "../data/boundaries/lineas_limite/SHP_ETRS89/recintos_provinciales_inspire_peninbal_etrs89/recintos_provinciales_inspire_peninbal_etrs89.shp"
)

provinces = provinces.to_crs(epsg=4326)
provinces['NAMEUNIT'] = provinces['NAMEUNIT'].str.strip()

provinces = provinces[
    provinces['NAMEUNIT'].isin(dataset_provinces)
].copy()

provinces[['NAMEUNIT']].sort_values('NAMEUNIT')

## 7. Load and filter municipal boundaries

In [ ]:
municipality_name_corrections = {
    'HUÉVAR': 'HUÉVAR DEL ALJARAFE',
    'LA GRANADA DEL RÍO TINTO': 'LA GRANADA DE RÍO-TINTO',
    'ZAHARA DE LA SIERRA': 'ZAHARA'
}

In [ ]:
municipalities = gpd.read_file(
    "../data/boundaries/lineas_limite/SHP_ETRS89/recintos_municipales_inspire_peninbal_etrs89/recintos_municipales_inspire_peninbal_etrs89.shp"
)

municipalities = municipalities.to_crs(epsg=4326)
municipalities['NAMEUNIT'] = municipalities['NAMEUNIT'].str.upper().str.strip()

municipalities_filtered_names = [
    municipality_name_corrections.get(m.upper().strip(), m.upper().strip())
    for m in dataset_municipalities
]

municipalities = municipalities[
    municipalities['NAMEUNIT'].isin(municipalities_filtered_names)
].copy()

print("Number of municipalities in dataset:", len(dataset_municipalities))
print("Number of filtered municipalities:", len(municipalities))

municipalities[['NAMEUNIT']].sort_values('NAMEUNIT')

In [ ]:
dataset_municipalities_upper = sorted([
    m.upper().strip()
    for m in dataset_municipalities
])

shape_municipalities_upper = sorted(
    municipalities['NAMEUNIT'].unique().tolist()
)

missing_in_shape = sorted(
    set(dataset_municipalities_upper) - set(shape_municipalities_upper)
)

print("Municipalities in dataset not matched in shapefile:")
print(missing_in_shape)

In [ ]:
municipalities_all = gpd.read_file(
    "../data/boundaries/lineas_limite/SHP_ETRS89/recintos_municipales_inspire_peninbal_etrs89/recintos_municipales_inspire_peninbal_etrs89.shp"
)

municipalities_all = municipalities_all.to_crs(epsg=4326)
municipalities_all['NAMEUNIT'] = municipalities_all['NAMEUNIT'].str.upper().str.strip()

In [ ]:
dataset_municipalities_upper = sorted([
    m.upper().strip()
    for m in dataset_municipalities
])

shape_municipalities_upper = sorted(
    municipalities_all['NAMEUNIT'].unique().tolist()
)

missing_in_shape = sorted(
    set(dataset_municipalities_upper) - set(shape_municipalities_upper)
)

print("Municipalities in dataset not matched in shapefile:")
print(missing_in_shape)

In [ ]:
[x for x in municipalities_all['NAMEUNIT'].unique() if 'HUÉVAR' in x]

In [ ]:
[x for x in municipalities_all['NAMEUNIT'].unique() if 'GRANADA' in x]

In [ ]:
[x for x in municipalities_all['NAMEUNIT'].unique() if 'ZAHARA' in x]

In [ ]:
andalusia = gpd.read_file(
    "../data/boundaries/lineas_limite/SHP_ETRS89/recintos_autonomicas_inspire_peninbal_etrs89/recintos_autonomicas_inspire_peninbal_etrs89.shp"
)

andalusia = andalusia.to_crs(epsg=4326)
andalusia['NAMEUNIT'] = andalusia['NAMEUNIT'].str.upper().str.strip()

andalusia = andalusia[
    andalusia['NAMEUNIT'] == 'ANDALUCÍA'
].copy()

andalusia[['NAMEUNIT']]

In [ ]:
provinces = gpd.read_file(
    "../data/boundaries/lineas_limite/SHP_ETRS89/recintos_provinciales_inspire_peninbal_etrs89/recintos_provinciales_inspire_peninbal_etrs89.shp"
)

provinces = provinces.to_crs(epsg=4326)
provinces['NAMEUNIT'] = provinces['NAMEUNIT'].str.upper().str.strip()

dataset_provinces = (
    df['province']
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .unique()
)

provinces = provinces[
    provinces['NAMEUNIT'].isin(dataset_provinces)
].copy()

print("Number of provinces in dataset:", len(dataset_provinces))
print("Number of filtered provinces:", len(provinces))

provinces[['NAMEUNIT']].sort_values('NAMEUNIT')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

andalusia.plot(
    ax=ax,
    color='white',
    edgecolor='black',
    linewidth=1.5
)

provinces.plot(
    ax=ax,
    color='lightgrey',
    edgecolor='black',
    linewidth=1
)

municipalities.plot(
    ax=ax,
    color='none',
    edgecolor='darkred',
    linewidth=0.5
)

ax.set_title('Study Area: Andalusia, Provinces and Municipalities', fontsize=16)
ax.axis('off')

plt.show()

## 8. Create georeferenced points dataset

In [ ]:
map_points = df[
    (df['latitude'].notna()) &
    (df['longitude'].notna()) &
    (df['latitude'] != 0) &
    (df['longitude'] != 0)
].copy()

print("Total rows in dataset:", len(df))
print("Georeferenced rows:", len(map_points))

map_points.head()

## 9. Convert georeferenced rows into a GeoDataFrame

In [ ]:
points_gdf = gpd.GeoDataFrame(
    map_points,
    geometry=gpd.points_from_xy(map_points['longitude'], map_points['latitude']),
    crs='EPSG:4326'
)

points_gdf.head()

## 10. Plot georeferenced points over Andalusia

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

andalusia.plot(
    ax=ax,
    color='white',
    edgecolor='black',
    linewidth=1.5
)

provinces.plot(
    ax=ax,
    color='lightgrey',
    edgecolor='black',
    linewidth=1
)

municipalities.plot(
    ax=ax,
    color='none',
    edgecolor='darkred',
    linewidth=0.4
)

points_gdf.plot(
    ax=ax,
    color='darkgreen',
    markersize=20,
    alpha=0.8
)

ax.set_title('Georeferenced felling locations', fontsize=16)
ax.axis('off')

plt.show()

## 11. Aggregate tree quantities by province

In [ ]:
province_totals = (
    df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

province_totals['province'] = (
    province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

province_totals = province_totals.sort_values(
    by='tree_quantity',
    ascending=False
)

province_totals

## 12. Aggregate tree quantities by municipality

In [ ]:
municipality_totals = (
    df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

municipality_totals['municipality'] = (
    municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

municipality_totals['municipality'] = municipality_totals['municipality'].replace(
    municipality_name_corrections
)

municipality_totals = municipality_totals.sort_values(
    by='tree_quantity',
    ascending=False
)

municipality_totals.head(20)

## 13. Merge province quantities with provincial polygons

In [ ]:
provinces_map = provinces.merge(
    province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

provinces_map['tree_quantity'] = provinces_map['tree_quantity'].fillna(0)

provinces_map[['NAMEUNIT', 'tree_quantity']].sort_values(
    by='tree_quantity',
    ascending=False
)

## 14. Merge municipality quantities with municipal polygons

In [ ]:
municipalities_map = municipalities.merge(
    municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

municipalities_map['tree_quantity'] = municipalities_map['tree_quantity'].fillna(0)

municipalities_map[['NAMEUNIT', 'tree_quantity']].sort_values(
    by='tree_quantity',
    ascending=False
).head(20)

## 15. Check matched polygons

In [ ]:
print("Matched provinces with data:")
print((provinces_map['tree_quantity'] > 0).sum())

print("Matched municipalities with data:")
print((municipalities_map['tree_quantity'] > 0).sum())

## 16. Static choropleth map by province

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

andalusia.plot(
    ax=ax,
    color='white',
    edgecolor='black',
    linewidth=1.5
)

provinces_map.plot(
    column='tree_quantity',
    cmap='OrRd',
    linewidth=1,
    edgecolor='black',
    legend=True,
    ax=ax
)

ax.set_title('Tree quantities by province', fontsize=16)
ax.axis('off')

plt.show()

## 17. Static choropleth map by municipality

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

andalusia.plot(
    ax=ax,
    color='white',
    edgecolor='black',
    linewidth=1.2
)

municipalities_map.plot(
    column='tree_quantity',
    cmap='OrRd',
    linewidth=0.4,
    edgecolor='black',
    legend=True,
    ax=ax
)

ax.set_title('Tree quantities by municipality', fontsize=16)
ax.axis('off')

plt.show()

## 18. Interactive choropleth map by province

In [ ]:
m_provinces = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

folium.Choropleth(
    geo_data=provinces_map,
    data=provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.5,
    legend_name='Tree quantities by province'
).add_to(m_provinces)

folium.GeoJson(
    provinces_map,
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_provinces)

m_provinces

In [ ]:
m_provinces.save("../img/maps/provinces_map.html")

## 19. Interactive choropleth map by municipality

In [ ]:
municipalities_map_filtered = municipalities_map[
    municipalities_map['tree_quantity'] > 0
].copy()

m_municipalities = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

folium.Choropleth(
    geo_data=municipalities_map_filtered,
    data=municipalities_map_filtered,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='OrRd',
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name='Tree quantities by municipality'
).add_to(m_municipalities)

folium.GeoJson(
    municipalities_map_filtered,
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_municipalities)

m_municipalities

In [ ]:
m_municipalities.save("../img/maps/municipalities_map.html")

## 20. Basic interactive point map

In [ ]:
m_points = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

for _, row in map_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_opacity=0.7
    ).add_to(m_points)

m_points

## 21. Cluster point map

In [ ]:
m_cluster = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

marker_cluster = MarkerCluster().add_to(m_cluster)

for _, row in map_points.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """
    ).add_to(marker_cluster)

m_cluster

## 22. Bubble map by tree quantity

In [ ]:
m_quantity = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

for _, row in map_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Landownership: {row['landownership']}<br>
        Landowner: {row['landowner_name']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkblue',
        fill=True,
        fill_opacity=0.6
    ).add_to(m_quantity)

m_quantity

## 23. Point map by landownership type

In [ ]:
landownership_colors = {
    'comunal': 'green',
    'private': 'blue',
    'noble estate': 'purple',
    'ecclesiastic': 'red',
    'assistance institution': 'orange',
    'fallow land': 'gray',
    'private, ecclesiastic': 'darkblue',
    'noble estate, ecclesiastic': 'darkred',
    'private, noble estate': 'cadetblue',
    'undefined': 'black'
}

In [ ]:
m_landownership = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

for _, row in map_points.iterrows():
    color = landownership_colors.get(row['landownership'], 'lightgray')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Landownership: {row['landownership']}<br>
        Landowner: {row['landowner_name']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color=color,
        fill=True,
        fill_opacity=0.7
    ).add_to(m_landownership)

m_landownership

## 24. Save interactive maps

In [ ]:
m_points.save("../img/maps/basic_points_map.html")
m_cluster.save("../img/maps/cluster_points_map.html")
m_quantity.save("../img/maps/bubble_quantity_map.html")
m_landownership.save("../img/maps/landownership_points_map.html")
m_provinces.save("../img/maps/provinces_map.html")
m_municipalities.save("../img/maps/municipalities_map.html")

## 25. Confirm saved files

In [ ]:
import os
print(os.listdir("../img/maps"))

## 26. Save static maps as PNG

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

andalusia.plot(
    ax=ax,
    color='white',
    edgecolor='black',
    linewidth=1.5
)

provinces_map.plot(
    column='tree_quantity',
    cmap='OrRd',
    linewidth=1,
    edgecolor='black',
    legend=True,
    ax=ax
)

ax.set_title('Tree quantities by province', fontsize=16)
ax.axis('off')

plt.savefig("../img/maps/provinces_map_static.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

andalusia.plot(
    ax=ax,
    color='white',
    edgecolor='black',
    linewidth=1.2
)

municipalities_map.plot(
    column='tree_quantity',
    cmap='OrRd',
    linewidth=0.4,
    edgecolor='black',
    legend=True,
    ax=ax
)

ax.set_title('Tree quantities by municipality', fontsize=16)
ax.axis('off')

plt.savefig("../img/maps/municipalities_map_static.png", dpi=300, bbox_inches="tight")
plt.show()

## 26. Interactive combined map: total tree quantities

## 27. Interactive combined maps by species

In [ ]:
m_total_combined = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

folium.Choropleth(
    geo_data=provinces_map,
    data=provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.35,
    line_opacity=0.8,
    legend_name='Tree quantities by province'
).add_to(m_total_combined)

folium.GeoJson(
    provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.2,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_total_combined)

folium.GeoJson(
    municipalities_map,
    style_function=lambda feature: {
        'fillColor': '#ff7800',
        'color': 'darkred',
        'weight': 0.5,
        'fillOpacity': 0.15
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_total_combined)

for _, row in map_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_opacity=0.7
    ).add_to(m_total_combined)

m_total_combined

In [ ]:
sorted(df['tree_species'].dropna().unique())

In [ ]:
species_selected = 'oak'

In [ ]:
species_df = df[
    df['tree_species'].str.lower() == species_selected.lower()
].copy()

species_points = species_df[
    (species_df['latitude'].notna()) &
    (species_df['longitude'].notna()) &
    (species_df['latitude'] != 0) &
    (species_df['longitude'] != 0)
].copy()

species_municipality_totals = (
    species_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

species_municipalities_map = municipalities.merge(
    species_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

species_municipalities_map['tree_quantity'] = (
    species_municipalities_map['tree_quantity']
    .fillna(0)
)

species_province_totals = (
    species_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

species_province_totals['province'] = (
    species_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_provinces_map = provinces.merge(
    species_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

species_provinces_map['tree_quantity'] = (
    species_provinces_map['tree_quantity']
    .fillna(0)
)

m_species = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=species_provinces_map,
    data=species_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{species_selected.title()} quantities by province'
).add_to(m_species)

# Tooltip das provincias
folium.GeoJson(
    species_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_species)

# Coropleto de municipios com a mesma escala de cores
folium.Choropleth(
    geo_data=species_municipalities_map,
    data=species_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{species_selected.title()} quantities by municipality'
).add_to(m_species)

# Tooltip dos municipios
folium.GeoJson(
    species_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_species)

# Pontos em verde
for _, row in species_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_species)

m_species

In [ ]:
m_species_oak = m_species

m_species_oak.save("../img/maps/species_oak_map.html")

In [ ]:
species_selected = 'holm oak'

In [ ]:
species_df = df[
    df['tree_species'].str.lower() == species_selected.lower()
].copy()

species_points = species_df[
    (species_df['latitude'].notna()) &
    (species_df['longitude'].notna()) &
    (species_df['latitude'] != 0) &
    (species_df['longitude'] != 0)
].copy()

species_municipality_totals = (
    species_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

species_municipalities_map = municipalities.merge(
    species_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

species_municipalities_map['tree_quantity'] = (
    species_municipalities_map['tree_quantity']
    .fillna(0)
)

species_province_totals = (
    species_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

species_province_totals['province'] = (
    species_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_provinces_map = provinces.merge(
    species_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

species_provinces_map['tree_quantity'] = (
    species_provinces_map['tree_quantity']
    .fillna(0)
)

m_species = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=species_provinces_map,
    data=species_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{species_selected.title()} quantities by province'
).add_to(m_species)

# Tooltip das provincias
folium.GeoJson(
    species_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_species)

# Coropleto de municipios com a mesma escala de cores
folium.Choropleth(
    geo_data=species_municipalities_map,
    data=species_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{species_selected.title()} quantities by municipality'
).add_to(m_species)

# Tooltip dos municipios
folium.GeoJson(
    species_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_species)

# Pontos em verde
for _, row in species_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_species)

m_species

In [ ]:
m_species_holm_oak = m_species

m_species_holm_oak.save("../img/maps/species_holm_oak_map.html")

In [ ]:
species_selected = 'gall oak'

In [ ]:
species_df = df[
    df['tree_species'].str.lower() == species_selected.lower()
].copy()

species_points = species_df[
    (species_df['latitude'].notna()) &
    (species_df['longitude'].notna()) &
    (species_df['latitude'] != 0) &
    (species_df['longitude'] != 0)
].copy()

species_municipality_totals = (
    species_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

species_municipalities_map = municipalities.merge(
    species_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

species_municipalities_map['tree_quantity'] = (
    species_municipalities_map['tree_quantity']
    .fillna(0)
)

species_province_totals = (
    species_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

species_province_totals['province'] = (
    species_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_provinces_map = provinces.merge(
    species_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

species_provinces_map['tree_quantity'] = (
    species_provinces_map['tree_quantity']
    .fillna(0)
)

m_species = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=species_provinces_map,
    data=species_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{species_selected.title()} quantities by province'
).add_to(m_species)

# Tooltip das provincias
folium.GeoJson(
    species_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_species)

# Coropleto de municipios com a mesma escala de cores
folium.Choropleth(
    geo_data=species_municipalities_map,
    data=species_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{species_selected.title()} quantities by municipality'
).add_to(m_species)

# Tooltip dos municipios
folium.GeoJson(
    species_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_species)

# Pontos em verde
for _, row in species_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_species)

m_species

In [ ]:
m_species_gall_oak = m_species

m_species_gall_oak.save("../img/maps/species_gall_oak_map.html")

In [ ]:
species_selected = 'pine'

In [ ]:
species_df = df[
    df['tree_species'].str.lower() == species_selected.lower()
].copy()

species_points = species_df[
    (species_df['latitude'].notna()) &
    (species_df['longitude'].notna()) &
    (species_df['latitude'] != 0) &
    (species_df['longitude'] != 0)
].copy()

species_municipality_totals = (
    species_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_municipality_totals['municipality'] = (
    species_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

species_municipalities_map = municipalities.merge(
    species_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

species_municipalities_map['tree_quantity'] = (
    species_municipalities_map['tree_quantity']
    .fillna(0)
)

species_province_totals = (
    species_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

species_province_totals['province'] = (
    species_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

species_provinces_map = provinces.merge(
    species_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

species_provinces_map['tree_quantity'] = (
    species_provinces_map['tree_quantity']
    .fillna(0)
)

m_species = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=species_provinces_map,
    data=species_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{species_selected.title()} quantities by province'
).add_to(m_species)

# Tooltip das provincias
folium.GeoJson(
    species_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_species)

# Coropleto de municipios com a mesma escala de cores
folium.Choropleth(
    geo_data=species_municipalities_map,
    data=species_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{species_selected.title()} quantities by municipality'
).add_to(m_species)

# Tooltip dos municipios
folium.GeoJson(
    species_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_species)

# Pontos em verde
for _, row in species_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_species)

m_species

In [ ]:
m_species_pine = m_species

m_species_pine.save("../img/maps/species_pine_map.html")

## 28. Interactive combined map by landownership type

In [ ]:
sorted(df['landownership'].dropna().unique())

In [ ]:
ownership_selected = 'comunal'

In [ ]:
ownership_df = df[
    df['landownership'].str.lower().str.contains(ownership_selected.lower(), na=False)
].copy()

ownership_points = ownership_df[
    (ownership_df['latitude'].notna()) &
    (ownership_df['longitude'].notna()) &
    (ownership_df['latitude'] != 0) &
    (ownership_df['longitude'] != 0)
].copy()

ownership_municipality_totals = (
    ownership_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

ownership_municipalities_map = municipalities.merge(
    ownership_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

ownership_municipalities_map['tree_quantity'] = (
    ownership_municipalities_map['tree_quantity']
    .fillna(0)
)

ownership_province_totals = (
    ownership_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

ownership_province_totals['province'] = (
    ownership_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_provinces_map = provinces.merge(
    ownership_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

ownership_provinces_map['tree_quantity'] = (
    ownership_provinces_map['tree_quantity']
    .fillna(0)
)

m_ownership = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=ownership_provinces_map,
    data=ownership_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{ownership_selected.title()} quantities by province'
).add_to(m_ownership)

# Tooltip das provincias
folium.GeoJson(
    ownership_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Coropleto de municipios
folium.Choropleth(
    geo_data=ownership_municipalities_map,
    data=ownership_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{ownership_selected.title()} quantities by municipality'
).add_to(m_ownership)

# Tooltip dos municipios
folium.GeoJson(
    ownership_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Pontos em verde
for _, row in ownership_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Landownership: {row['landownership']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_ownership)

m_ownership

In [ ]:
m_ownership_comunal = m_ownership

m_ownership_comunal.save("../img/maps/ownership_comunal_map.html")

In [ ]:
ownership_selected = 'noble estate'

In [ ]:
ownership_df = df[
    df['landownership'].str.lower().str.contains(ownership_selected.lower(), na=False)
].copy()

ownership_points = ownership_df[
    (ownership_df['latitude'].notna()) &
    (ownership_df['longitude'].notna()) &
    (ownership_df['latitude'] != 0) &
    (ownership_df['longitude'] != 0)
].copy()

ownership_municipality_totals = (
    ownership_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

ownership_municipalities_map = municipalities.merge(
    ownership_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

ownership_municipalities_map['tree_quantity'] = (
    ownership_municipalities_map['tree_quantity']
    .fillna(0)
)

ownership_province_totals = (
    ownership_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

ownership_province_totals['province'] = (
    ownership_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_provinces_map = provinces.merge(
    ownership_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

ownership_provinces_map['tree_quantity'] = (
    ownership_provinces_map['tree_quantity']
    .fillna(0)
)

m_ownership = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=ownership_provinces_map,
    data=ownership_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{ownership_selected.title()} quantities by province'
).add_to(m_ownership)

# Tooltip das provincias
folium.GeoJson(
    ownership_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Coropleto de municipios
folium.Choropleth(
    geo_data=ownership_municipalities_map,
    data=ownership_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{ownership_selected.title()} quantities by municipality'
).add_to(m_ownership)

# Tooltip dos municipios
folium.GeoJson(
    ownership_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Pontos em verde
for _, row in ownership_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Landownership: {row['landownership']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_ownership)

m_ownership

In [ ]:
m_ownership_noble_estate = m_ownership

m_ownership_noble_estate.save("../img/maps/ownership_noble_estate_map.html")

In [ ]:
ownership_selected = 'private'

In [ ]:
ownership_df = df[
    df['landownership'].str.lower().str.contains(ownership_selected.lower(), na=False)
].copy()

ownership_points = ownership_df[
    (ownership_df['latitude'].notna()) &
    (ownership_df['longitude'].notna()) &
    (ownership_df['latitude'] != 0) &
    (ownership_df['longitude'] != 0)
].copy()

ownership_municipality_totals = (
    ownership_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

ownership_municipalities_map = municipalities.merge(
    ownership_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

ownership_municipalities_map['tree_quantity'] = (
    ownership_municipalities_map['tree_quantity']
    .fillna(0)
)

ownership_province_totals = (
    ownership_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

ownership_province_totals['province'] = (
    ownership_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_provinces_map = provinces.merge(
    ownership_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

ownership_provinces_map['tree_quantity'] = (
    ownership_provinces_map['tree_quantity']
    .fillna(0)
)

m_ownership = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=ownership_provinces_map,
    data=ownership_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{ownership_selected.title()} quantities by province'
).add_to(m_ownership)

# Tooltip das provincias
folium.GeoJson(
    ownership_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Coropleto de municipios
folium.Choropleth(
    geo_data=ownership_municipalities_map,
    data=ownership_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{ownership_selected.title()} quantities by municipality'
).add_to(m_ownership)

# Tooltip dos municipios
folium.GeoJson(
    ownership_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Pontos em verde
for _, row in ownership_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Landownership: {row['landownership']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_ownership)

m_ownership

In [ ]:
m_ownership_private = m_ownership

m_ownership_private.save("../img/maps/ownership_private_map.html")

In [ ]:
ownership_selected = 'ecclesiastic'

In [ ]:
ownership_df = df[
    df['landownership'].str.lower().str.contains(ownership_selected.lower(), na=False)
].copy()

ownership_points = ownership_df[
    (ownership_df['latitude'].notna()) &
    (ownership_df['longitude'].notna()) &
    (ownership_df['latitude'] != 0) &
    (ownership_df['longitude'] != 0)
].copy()

ownership_municipality_totals = (
    ownership_df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_municipality_totals['municipality'] = (
    ownership_municipality_totals['municipality']
    .replace(municipality_name_corrections)
)

ownership_municipalities_map = municipalities.merge(
    ownership_municipality_totals,
    left_on='NAMEUNIT',
    right_on='municipality',
    how='left'
)

ownership_municipalities_map['tree_quantity'] = (
    ownership_municipalities_map['tree_quantity']
    .fillna(0)
)

ownership_province_totals = (
    ownership_df.groupby('province', as_index=False)['tree_quantity']
    .sum()
)

ownership_province_totals['province'] = (
    ownership_province_totals['province']
    .astype(str)
    .str.upper()
    .str.strip()
)

ownership_provinces_map = provinces.merge(
    ownership_province_totals,
    left_on='NAMEUNIT',
    right_on='province',
    how='left'
)

ownership_provinces_map['tree_quantity'] = (
    ownership_provinces_map['tree_quantity']
    .fillna(0)
)

m_ownership = folium.Map(
    location=[37.4, -4.8],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Coropleto de provincias
folium.Choropleth(
    geo_data=ownership_provinces_map,
    data=ownership_provinces_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.20,
    line_opacity=0.4,
    legend_name=f'{ownership_selected.title()} quantities by province'
).add_to(m_ownership)

# Tooltip das provincias
folium.GeoJson(
    ownership_provinces_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1.0,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Province:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Coropleto de municipios
folium.Choropleth(
    geo_data=ownership_municipalities_map,
    data=ownership_municipalities_map,
    columns=['NAMEUNIT', 'tree_quantity'],
    key_on='feature.properties.NAMEUNIT',
    fill_color='YlOrRd',
    fill_opacity=0.45,
    line_opacity=0.5,
    legend_name=f'{ownership_selected.title()} quantities by municipality'
).add_to(m_ownership)

# Tooltip dos municipios
folium.GeoJson(
    ownership_municipalities_map,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#7f2704',
        'weight': 0.6,
        'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAMEUNIT', 'tree_quantity'],
        aliases=['Municipality:', 'Tree quantity:']
    )
).add_to(m_ownership)

# Pontos em verde
for _, row in ownership_points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(3, row['tree_quantity'] / 50),
        popup=f"""
        Place: {row['felling_toponymy_source']}<br>
        Province: {row['province']}<br>
        Municipality: {row['municipality']}<br>
        Landownership: {row['landownership']}<br>
        Species: {row['tree_species']}<br>
        Quantity: {row['tree_quantity']}
        """,
        color='darkgreen',
        fill=True,
        fill_color='green',
        fill_opacity=0.75
    ).add_to(m_ownership)

m_ownership

In [ ]:
m_ownership_ecclesiastic = m_ownership

m_ownership_ecclesiastic.save("../img/maps/ownership_ecclesiastic_map.html")

In [ ]:
m_total_combined.save("../img/maps/total_combined_map.html")

## 29. Confirm saved maps

In [ ]:
import os

sorted(os.listdir("../img/maps"))

## 30. Quick summary tables

In [ ]:
province_summary = (
    df.groupby('province', as_index=False)['tree_quantity']
    .sum()
    .sort_values(by='tree_quantity', ascending=False)
)

province_summary

In [ ]:
municipality_summary = (
    df.groupby('municipality', as_index=False)['tree_quantity']
    .sum()
    .sort_values(by='tree_quantity', ascending=False)
    .head(15)
)

municipality_summary

In [ ]:
species_summary = (
    df.groupby('tree_species', as_index=False)['tree_quantity']
    .sum()
    .sort_values(by='tree_quantity', ascending=False)
)

species_summary

In [ ]:
ownership_summary = (
    df.groupby('landownership', as_index=False)['tree_quantity']
    .sum()
    .sort_values(by='tree_quantity', ascending=False)
)

ownership_summary

## 31. Summary

## Main spatial analysis conclusions

- The study area includes four Andalusian provinces represented in the dataset.
- Spatial analysis was carried out at three levels: province, municipality and point location.
- Interactive maps were created for total tree quantities, species distribution and landownership types.
- Choropleth maps were combined with point-based maps to visualise both aggregate patterns and individual felling locations.
- Pine, oak, holm oak and gall oak were selected as the main species categories.
- Comunal, noble estate, private and ecclesiastic ownership types were selected as the main landownership categories.
- All interactive maps were exported as HTML files for later use in the Streamlit application.